In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType, ArrayType

import time
import io
from pyspark.sql import functions as F
from datetime import datetime
from datasets import load_dataset
import warnings
import os
from pathlib import Path
import json
import requests

warnings.filterwarnings('ignore')
%load_ext sparksql_magic


builder = SparkSession.builder \
    .appName("DeltaLakeLocal") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Automatically downloads the correct matching Delta jars
spark = configure_spark_with_delta_pip(builder).getOrCreate()
%config SparkSqlMagic.spark = spark

/home/obaliuta/miniconda3/envs/spark_with_delta/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
your 131072x1 screen size is bogus. expect trouble
26/05/23 13:51:16 WARN Utils: Your hostname, DESKTOP-IS86RQE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/23 13:51:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/obaliuta/miniconda3/envs/spark_with_delta/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/obaliuta/.ivy2/cache
The jars for the packages stored in: /home/obaliuta/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3a02bed3-e71f-4102-9e08-a381d824974d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 272ms :: artifacts dl 11ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |

26/05/23 13:51:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
#getting the data
dataset = load_dataset("tchebonenko/MedicalTranscriptions")
df = dataset['train'].to_pandas() if 'train' in dataset else dataset.to_pandas()

Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 100%|██████████| 4999/4999 [00:01<00:00, 3035.75 examples/s]


In [34]:
#getting only full medical description and limit to 50 line for the test

summary_data = pd.DataFrame(df['transcription'].head(50))

In [35]:
#openai params

api_key = os.environ.get('OPEN_AI_KEY')
if not api_key:
    key_file = Path.home() / '.config' / 'open_ai' / 'key'
    if key_file.exists():
        api_key = key_file.read_text().strip()
    else:
        raise FileNotFoundError("API key not found. Set OPEN_AI_KEY or run scripts/store_key.sh <key>")

model = "gpt-4o-mini"
ai_url = "https://api.openai.com/v1/chat/completions"        

In [36]:
def medical_assist(text):
    """Structurize medical data from the given information."""
    if not text or text.strip() == "":
        return text
    
    system_prompt = f"You are a medical assistant. Structurize the following medical information. Return ONLY the structured data in JSON format, no explanations. If the information is incomplete, return what you can and leave missing fields empty. The JSON should have the following format: {{'symptoms': [], 'diagnosis': '', 'medical_specialty': '', 'treatment': ''}}. Here is the information: {text}"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        "temperature": 0.8
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    try:
        response = requests.post(ai_url, headers=headers, json=payload)
        response.raise_for_status()

        result = response.json()
        output = result['choices'][0]['message']['content'].strip()
        
        
        return output
    except Exception as e:
        print(f"Processing error: {e}")
        return text  # Return original text on error

In [37]:
summary_data['processed'] = summary_data['transcription'].apply(medical_assist) 

In [40]:
df_spark = spark.createDataFrame(summary_data)



# Define the JSON schema matching the processed field structure
processed_schema = StructType([
    StructField("symptoms", ArrayType(StringType()), True),
    StructField("diagnosis", StringType(), True),
    StructField("medical_specialty", StringType(), True),
    StructField("treatment", StringType(), True)
])

# Strip markdown code fences if present, parse JSON, then expand into columns
df_expanded = df_spark \
    .withColumn(
        "processed_clean",
        F.regexp_replace(F.col("processed"), r"```json\s*|\s*```", "")
    ) \
    .withColumn("processed_json", F.from_json(F.col("processed_clean"), processed_schema)) \
    .withColumn("symptoms", F.col("processed_json.symptoms")) \
    .withColumn("diagnosis", F.col("processed_json.diagnosis")) \
    .withColumn("medical_specialty", F.col("processed_json.medical_specialty")) \
    .withColumn("treatment", F.col("processed_json.treatment")) \
    .drop("processed", "processed_clean", "processed_json")


In [42]:
df_expanded.printSchema()

root
 |-- transcription: string (nullable = true)
 |-- symptoms: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- diagnosis: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- treatment: string (nullable = true)



In [ ]:
home_dir = os.path.expanduser("~")
delta_base_path = os.path.join(home_dir, "lakehouse_test/medical_assist")

df_expanded.write.format("delta").option("mergeSchema", "true").mode("append").save(delta_base_path+"/silver/dataset")

In [4]:
%%sparksql
select *
from delta.`/home/obaliuta/lakehouse_test/medical_assist/silver/dataset`
limit 5